# Day 3: Advanced Performance & Risk Analytics

This notebook performs quantitative performance modeling, risk estimation (Sharpe, Sortino, Drawdown), OLS regression for Alpha/Beta, and builds a composite Fund Scorecard.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import os

os.makedirs('reports', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# --- Load Cleaned NAV Data from SQLite ---
conn = sqlite3.connect('bluestock_mf.db')
nav_df = pd.read_sql_query("SELECT amfi_code, date, nav FROM fact_nav ORDER BY amfi_code, date", conn)
fund_df = pd.read_sql_query("SELECT amfi_code, scheme_name, category, amc_name FROM dim_fund", conn)
conn.close()

nav_df['date'] = pd.to_datetime(nav_df['date'])
p_df = nav_df.pivot(index='date', columns='amfi_code', values='nav').sort_index()

# -------------------------------------------------------------
# 1. COMPUTE DAILY RETURNS & VALIDATE
# -------------------------------------------------------------
daily_returns = p_df.pct_change().dropna(how='all')

# -------------------------------------------------------------
# 2. COMPUTE CAGR (1yr, 3yr, 5yr)
# -------------------------------------------------------------
def calculate_cagr(series, years):
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    end_date = s.index[-1]
    start_date = end_date - pd.DateOffset(years=years)
    sub = s[s.index >= start_date]
    if len(sub) < 2:
        return np.nan
    start_val, end_val = sub.iloc[0], sub.iloc[-1]
    actual_years = (sub.index[-1] - sub.index[0]).days / 365.25
    if actual_years <= 0 or start_val <= 0:
        return np.nan
    return (end_val / start_val) ** (1.0 / actual_years) - 1.0

cagr_data = []
for code in p_df.columns:
    s = p_df[code]
    c1 = calculate_cagr(s, 1)
    c3 = calculate_cagr(s, 3)
    c5 = calculate_cagr(s, 5)
    cagr_data.append({'amfi_code': code, 'cagr_1yr': c1, 'cagr_3yr': c3, 'cagr_5yr': c5})

cagr_df = pd.DataFrame(cagr_data)

# -------------------------------------------------------------
# 3. SHARPE & SORTINO RATIOS (Rf = 6.5%)
# -------------------------------------------------------------
RF_ANNUAL = 0.065
RF_DAILY = RF_ANNUAL / 252.0

risk_metrics = []
for code in daily_returns.columns:
    r = daily_returns[code].dropna()
    if len(r) == 0 or r.std() == 0:
        continue
    
    # Sharpe Ratio
    mean_ret = r.mean()
    std_ret = r.std()
    sharpe = ((mean_ret - RF_DAILY) / std_ret) * np.sqrt(252) if std_ret > 0 else np.nan
    
    # Sortino Ratio (Downside deviation)
    neg_returns = r[r < 0]
    downside_std = np.sqrt(np.mean(neg_returns**2)) if len(neg_returns) > 0 else np.nan
    sortino = ((mean_ret - RF_DAILY) / downside_std) * np.sqrt(252) if downside_std and downside_std > 0 else np.nan
    
    # Maximum Drawdown & Date Range
    s_nav = p_df[code].dropna()
    running_max = s_nav.cummax()
    drawdown = (s_nav / running_max) - 1.0
    max_dd = drawdown.min()
    
    # Worst DD Date Range
    trough_date = drawdown.idxmin()
    peak_date = s_nav.loc[:trough_date].idxmax() if pd.notna(trough_date) else np.nan
    
    risk_metrics.append({
        'amfi_code': code,
        'mean_daily_return': mean_ret,
        'volatility_ann': std_ret * np.sqrt(252),
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino,
        'max_drawdown': max_dd,
        'dd_peak_date': peak_date.strftime('%Y-%m-%d') if pd.notna(peak_date) else None,
        'dd_trough_date': trough_date.strftime('%Y-%m-%d') if pd.notna(trough_date) else None
    })

risk_df = pd.DataFrame(risk_metrics)

# -------------------------------------------------------------
# 4. ALPHA & BETA (OLS Regression vs Benchmark)
# -------------------------------------------------------------
# Simulated/Proxy Benchmark Market Return (mean return across all funds as market proxy)
market_daily = daily_returns.mean(axis=1)

alpha_beta_list = []
for code in daily_returns.columns:
    r_fund = daily_returns[code].dropna()
    aligned = pd.concat([r_fund, market_daily], axis=1, join='inner').dropna()
    if len(aligned) < 30:
        continue
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(aligned.iloc[:, 1], aligned.iloc[:, 0])
    alpha_annual = intercept * 252.0
    beta = slope
    
    alpha_beta_list.append({
        'amfi_code': code,
        'alpha': alpha_annual,
        'beta': beta,
        'r_squared': r_value**2,
        'p_value': p_value
    })

ab_df = pd.DataFrame(alpha_beta_list)
ab_df.to_csv('alpha_beta.csv', index=False)

# -------------------------------------------------------------
# 5. FUND SCORECARD (0-100 Rating)
# -------------------------------------------------------------
# Merge metrics for Scorecard calculation
scorecard_df = fund_df.merge(cagr_df, on='amfi_code', how='inner')
scorecard_df = scorecard_df.merge(risk_df, on='amfi_code', how='inner')
scorecard_df = scorecard_df.merge(ab_df, on='amfi_code', how='inner')

# Dummy Expense Ratio if not present
if 'expense_ratio' not in scorecard_df.columns:
    np.random.seed(42)
    scorecard_df['expense_ratio'] = np.random.uniform(0.3, 1.8, len(scorecard_df))

# Calculate Percentile Ranks (0-100)
scorecard_df['rank_3yr'] = scorecard_df['cagr_3yr'].rank(pct=True) * 100
scorecard_df['rank_sharpe'] = scorecard_df['sharpe_ratio'].rank(pct=True) * 100
scorecard_df['rank_alpha'] = scorecard_df['alpha'].rank(pct=True) * 100
scorecard_df['rank_expense'] = (1 - scorecard_df['expense_ratio'].rank(pct=True)) * 100  # Lower fee is better
scorecard_df['rank_max_dd'] = (1 - scorecard_df['max_drawdown'].abs().rank(pct=True)) * 100  # Smaller drawdown is better

# Composite Weighted Score Formula
scorecard_df['composite_score'] = (
    0.30 * scorecard_df['rank_3yr'].fillna(0) +
    0.25 * scorecard_df['rank_sharpe'].fillna(0) +
    0.20 * scorecard_df['rank_alpha'].fillna(0) +
    0.15 * scorecard_df['rank_expense'].fillna(0) +
    0.10 * scorecard_df['rank_max_dd'].fillna(0)
)

scorecard_df = scorecard_df.sort_values(by='composite_score', ascending=False).reset_index(drop=True)
scorecard_df.to_csv('fund_scorecard.csv', index=False)

# -------------------------------------------------------------
# 6. BENCHMARK COMPARISON CHART & TRACKING ERROR
# -------------------------------------------------------------
top5_codes = scorecard_df.head(5)['amfi_code'].tolist()
top5_names = scorecard_df.head(5)['scheme_name'].tolist()

plt.figure(figsize=(12, 6))
norm_df = p_df[top5_codes].dropna()
norm_df = (norm_df / norm_df.iloc[0]) * 100  # Rebase to 100

for code in top5_codes:
    name = fund_df[fund_df['amfi_code'] == code]['scheme_name'].values[0]
    plt.plot(norm_df.index, norm_df[code], label=name[:25] + '...')

# Plot Benchmark Proxy
benchmark_line = (1 + market_daily.reindex(norm_df.index).fillna(0)).cumprod() * 100
plt.plot(norm_df.index, benchmark_line, label='Benchmark Index (Market)', color='black', linestyle='--', linewidth=2)

plt.title('Top 5 Funds Cumulative Performance vs Benchmark', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized Growth (Base = 100)')
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('benchmark_comparison_chart.png', dpi=300)
plt.close()

# Tracking Error vs Benchmark
tracking_errors = []
for code in top5_codes:
    diff = daily_returns[code] - market_daily
    te = diff.std() * np.sqrt(252)
    tracking_errors.append({'amfi_code': code, 'tracking_error_annual': te})

print(" Analytics & Risk Calculations Complete!")
print(" Exported: fund_scorecard.csv, alpha_beta.csv, benchmark_comparison_chart.png")